Napredna analiza podataka - Projekat - Klasterizacija teksta

Detekcija Phising mejlova na osnovu sadržaja mejla

Uvod: Razraditi

Instalacija potrebnih biblioteka

In [2]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, GridSearchCV
import matplotlib.pyplot as plt
import spacy

Prvobitno ucitavanje dataset-a sa dodatnim informacijama o broju unikatnih vrednosti opservacija i NaN  vrednosti

In [4]:
data = pd.read_csv('./data/emails/CEAS_08.csv')
data

,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1
...,...,...,...,...,...,...,...
39149,CNN Alerts <charlene-detecton@btcmarketing.com>,email1007@gvc.ceas-challenge.cc,"Fri, 08 Aug 2008 10:34:50 -0400",CNN Alerts: My Custom Alert,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,0
39150,CNN Alerts <idgetily1971@careplusnj.org>,email104@gvc.ceas-challenge.cc,"Fri, 08 Aug 2008 10:35:11 -0400",CNN Alerts: My Custom Alert,\n\nCNN Alerts: My Custom Alert\n\n\n\n\n\n\n ...,1,0
39151,Abhijit Vyas <xpojhbz@gmail.com>,fxgmqwjn@triptracker.net,"Fri, 08 Aug 2008 22:00:43 +0800",Slideshow viewer,Hello there ! \nGreat work on the slide show v...,0,0
39152,Joseph Brennan <vupzesm@columbia.edu>,zqoqi@spamassassin.apache.org,"Fri, 08 Aug 2008 09:00:46 -0500",Note on 2-digit years,"\nMail from sender , coming from intuit.com\ns...",0,0


U datom datasetu postoje sledece promenljive:
Sender - Naziv posiljaoca  i mejl adresa
Receiver - mejl adresa primaoca
Date - datum slanja
Subject - naslov mejla
Body - sadrzaj mejla
Label - binarna promenljiva koja oznacava da li mejl predstavlja phishing napad ili ne. 1 je phishing dok 0 nije.
Urls - binarna promenljiva koja oznacava da li se u mejlu nalaze url-ovi

Prvo sto cu da proverim je ucestalost vrednosti promenljivih koje u sebi sadrze NA vrednosti

In [6]:
data.receiver.value_counts()

receiver
user6@gvc.ceas-challenge.cc                                                          1375
wkilxloc@opensuse.org                                                                1230
user2.1@gvc.ceas-challenge.cc                                                        1037
user2.2@gvc.ceas-challenge.cc                                                         922
user2.4@gvc.ceas-challenge.cc                                                         738
                                                                                     ... 
vegetahan@hotmail.com                                                                   1
'Squirrelmail Developers Mailing List' <aqcvnwkdyeem-hbpoa@lists.sourceforge.net>       1
Frank Steiner <lwvrlirx-chvz0@bio.ifi.lmu.de>                                           1
Olyvia Meyer <kwycqr.unjnu@gmail.com>, Samantha Gilmour <orribdffovqb@gmail.com>        1
rgkwpdahmhmail@cs.cmu.edu, faq@engr.orst.edu, ji-ivay@googlegroups.com                  1
N

Kao sto se moze videti, veliki broj mejlova ima iste primaoce, pretpostavljam da su ovi mejlovi sluzili kao honeypot-ovi za prikupljanje phishing napada. Podaci primaoca se onda tesko mogu koristiti za

Postoji rizik od overfitting-a, ukoliko treniram model na ovim podacima. Jedino za sta bi mogla da mi sluzi promenljiva primaoc je da se na primer poredi adresa primaoca i posiljaoca. Ukoliko su mejl serveri isti, onda postoji veca sansa da je mejl validan

Probacu da nadjem mejlove(opservacije) koje sadrze isti mejl servera posiljaoca i primaoca

In [14]:
sender_domain = data['sender'].str.split("@").str[1].str.removesuffix(">")
receiver_domain = data['receiver'].str.split("@").str[1].str.removesuffix(">")

internal_sending_data = data.loc[sender_domain == receiver_domain]
internal_sending_data

,sender,receiver,date,subject,body,label,urls
22,ohwwlejs-pbfotbp@list.scms.waikato.ac.nz,iuxxdkqk@list.scms.waikato.ac.nz,"Tue, 05 Aug 2008 18:32:19 -0500","Wekalist Digest, Vol 60, Issue 10",Send Wekalist mailing list submissions to\n\ti...,0,1
54,Perl Jobs <xycn-vtnhz@perl.org>,necc@perl.org,"Tue, 05 Aug 2008 02:47:22 -0800","[Perl Jobs] Perl developer, Hungary, Budapest",Online URL for this job: http://jobs.perl.org/...,0,1
55,Perl Jobs <xycn-vtnhz@perl.org>,necc@perl.org,"Tue, 05 Aug 2008 02:48:34 -0800","[Perl Jobs] Software Engineer, Weather (onsite...",Online URL for this job: http://jobs.perl.org/...,0,1
57,Perl Jobs <xycn-vtnhz@perl.org>,necc@perl.org,"Tue, 05 Aug 2008 02:47:32 -0800",[Perl Jobs] Perl & Apache & Unix/Linux & Mysql...,Online URL for this job: http://jobs.perl.org/...,0,1
59,Perl Jobs <xycn-vtnhz@perl.org>,necc@perl.org,"Tue, 05 Aug 2008 08:39:46 -0800","[Perl Jobs] Perl Developer (onsite), United St...",Online URL for this job: http://jobs.perl.org/...,0,1
...,...,...,...,...,...,...,...
38861,"""Alexander Gelbukh (MICAI-2008)"" <fwj2317t@mic...",'AI group' <aqz4438d@micai.org>,"Fri, 08 Aug 2008 05:43:21 -0600",[UAI] CFP: MICAI 2008: Artifiical Intelligence...,MICAI 2008: 7th Mexican International Conferen...,0,0
38880,ysxshwee-tyrxx-liyysxs@lists.tuxonice.net,enpsfhxz-stvnz@lists.tuxonice.net,"Fri, 08 Aug 2008 13:33:35 +0000","TuxOnIce-users Digest, Vol 37, Issue 8",Send TuxOnIce-users mailing list submissions t...,0,1
38885,ysxshwee-tyrxx-liyysxs@lists.tuxonice.net,enpsfhxz-stvnz@lists.tuxonice.net,"Fri, 08 Aug 2008 13:33:56 +0000","TuxOnIce-users Digest, Vol 37, Issue 8",Send TuxOnIce-users mailing list submissions t...,0,1
38959,Georg Brandl <splmn@python.org>,zttpnd-wnlugsli@python.org,"Fri, 08 Aug 2008 15:14:13 +0200","Pygments 0.10 ""Malzeug"" released",I've just uploaded the Pygments 0.10 packages ...,0,1


In [20]:
internal_sent = internal_sending_data.loc[internal_sending_data['label'] == 0]
print(f"Ukupan broj internih mejlova koji nisu phising: {len(internal_sent)} od {len(internal_sending_data)}")
#TODO: Kreirati promenljivu is_internal

Ukupan broj internih mejlova koji nisu phising: 1226 od 1263


Kao sto se moze videti veliki broj izdvojenih mejlova su zaista rezultat internog slanja. Predstavljaju oko 3% internog slanja, nije veliki broj ovakvih opservacija ali predstavljaju bitan podataka za povecanje sigurnosti bezbednog mejla. Moguce je izvrsiti transformaciju gde ce se dodati binarna promenljiva is_internal. Zakljucujem da postoji velika korelacija izmedju poklapanja serverskih domena i status phising-a

Sledece sto se moze proveriti je da li kako izgledaju opservacije koje nemaju specificiranog primaoca, ovakvi mejlovi obicno predstavljaju pretnju ako su mejlovi samo navedeni u bcc

In [22]:
no_recipient_mails = data.loc[data['receiver'].isna() == True]
no_recipient_mails

,sender,receiver,date,subject,body,label,urls
294,Tony Earnshaw <uhbyq@hetnet.nl>,NaN,"Wed, 06 Aug 2008 00:47:37 +0100",Re: [dkim-milter-discuss] postfix + dkim-filte...,"Daniel Black skrev, on 08-02-2008 13:19:\n> On...",0,0
335,Tony Earnshaw <uhbyq@hetnet.nl>,NaN,"Wed, 06 Aug 2008 00:49:33 +0100",Re: [dkim-milter-discuss] postfix + dkim-filte...,"Wietse Venema skrev, on 08-02-2008 14:48:\n> T...",0,0
601,mouss <qpeqr@netoyen.net>,NaN,"Wed, 06 Aug 2008 01:04:34 +0100",Re: permission denied while using pipe in mast...,"Jordi Moles wrote:\n> Hi,\n>\n> the shellscrip...",0,0
611,mouss <qpeqr@netoyen.net>,NaN,"Wed, 06 Aug 2008 01:04:58 +0100",Re: Forwarding all localmail,Randy Ramsdell wrote:\n> I would like to know ...,0,0
620,mouss <qpeqr@netoyen.net>,NaN,"Wed, 06 Aug 2008 01:05:33 +0100",Re: SASL questions,"AlxFrag wrote:\n> Hi,\n>\n> i have a couple of...",0,0
...,...,...,...,...,...,...,...
38953,mouss <qpeqr@netoyen.net>,NaN,"Fri, 08 Aug 2008 14:41:20 +0100",Re: Challenging Local Sender/Recipient Spam,Gary C. New wrote:\n> We are having an issue w...,0,0
38971,mouss <qpeqr@netoyen.net>,NaN,"Fri, 08 Aug 2008 14:43:38 +0100",Re: failover for check_policy_service,"Alex wrote:\n> Hi,\n>\n> i've a question relat...",0,0
38975,mouss <qpeqr@netoyen.net>,NaN,"Fri, 08 Aug 2008 14:44:24 +0100",Re: Question about changing the road of authen...,"Johnny Maillist wrote:\n> Hi,\n> I want to cha...",0,0
38998,mouss <qpeqr@netoyen.net>,NaN,"Fri, 08 Aug 2008 14:46:11 +0100","Re: Lots of ""scam"" messages getting through SA","Robert S wrote:\n> I have started, over the la...",0,0


In [24]:
no_recipient_mails_ham = no_recipient_mails.loc[no_recipient_mails['label'] == 0]
print(f"Ukupan broj mejlova bez primaoca koji nisu phising: {len(no_recipient_mails_ham)} od {len(no_recipient_mails)}")

Ukupan broj mejlova bez primaoca koji nisu phising: 459 od 462


Moja pocetna tvrdnja ipak nije tacna, prema ovom dataset-u vecina mejlova koja nema primaoca nije bila phishing